# 06b — Supplementary Evaluation

**Input:** `../data/processed/pm_day_features.csv`, `../outputs/tables/oof_*.csv`
**Output:** `../outputs/tables/brier_scores.csv`, `../outputs/tables/ppv_npv_at_alert_rates.csv`, `../outputs/tables/confound_checks.csv`

**Description:**
- Brier scores for all OOF predictions
- PPV/NPV at fixed alert rates (top 1%, 2%, 5%, 10%)
- Pearson/Spearman OOF correlations with continuous DSI
- Sign-flip / confound checks: instability vs word count correlation
- Matches original cells 17-18, 31, 32, 42

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
from sklearn.metrics import brier_score_loss

# =========================
# CONFIG
# =========================
DATA_PATH = os.path.join("..", "data", "processed", "pm_day_features.csv")
OOF_DIR = os.path.join("..", "outputs", "tables")
OUT_DIR = OOF_DIR

PID_COL = "expiwell_id_clean"
CRISIS_COL = "crisis_PM_from_full"

ALERT_FRACS = [0.01, 0.02, 0.05, 0.10]

In [ ]:
# =========================
# LOAD
# =========================
pm_day = pd.read_csv(DATA_PATH)
print("Loaded feature data:", pm_day.shape)

In [ ]:
# =========================
# BRIER SCORES FOR ALL OOF FILES
# =========================
oof_files = sorted(glob.glob(os.path.join(OOF_DIR, "oof_*.csv")))
print(f"Found {len(oof_files)} OOF files")

brier_rows = []
for f in oof_files:
    label = os.path.basename(f).replace("oof_", "").replace(".csv", "")
    df = pd.read_csv(f)

    if "y" not in df.columns:
        continue

    y = df["y"].astype(int).values

    for model_col in ["p_base", "p_full"]:
        if model_col not in df.columns:
            continue
        p = df[model_col].values
        mask = np.isfinite(p)
        if mask.sum() < 10:
            continue
        bs = brier_score_loss(y[mask], p[mask])
        brier_rows.append({"label": label, "model": model_col, "brier_score": bs, "n": int(mask.sum())})

brier_df = pd.DataFrame(brier_rows)
brier_path = os.path.join(OUT_DIR, "brier_scores.csv")
brier_df.to_csv(brier_path, index=False)
print("\nBrier scores:")
print(brier_df.to_string(index=False))
print("\nSaved:", brier_path)

In [ ]:
# =========================
# PPV / NPV AT FIXED ALERT RATES
# =========================
ppv_rows = []

for f in oof_files:
    label = os.path.basename(f).replace("oof_", "").replace(".csv", "")
    df = pd.read_csv(f)
    if "y" not in df.columns:
        continue

    y = df["y"].astype(int).values

    for model_col in ["p_base", "p_full"]:
        if model_col not in df.columns:
            continue
        p = df[model_col].values
        order = np.argsort(-p)

        for frac in ALERT_FRACS:
            k = max(1, int(np.ceil(frac * len(y))))
            top_idx = order[:k]
            bot_idx = order[k:]

            tp = int(y[top_idx].sum())
            fp = k - tp
            fn = int(y[bot_idx].sum())
            tn = len(bot_idx) - fn

            ppv = tp / k if k > 0 else np.nan
            npv = tn / len(bot_idx) if len(bot_idx) > 0 else np.nan
            recall = tp / max(1, int(y.sum()))

            ppv_rows.append({
                "label": label, "model": model_col, "alert_frac": frac,
                "k": k, "TP": tp, "FP": fp, "FN": fn, "TN": tn,
                "PPV": ppv, "NPV": npv, "recall": recall,
            })

ppv_df = pd.DataFrame(ppv_rows)
ppv_path = os.path.join(OUT_DIR, "ppv_npv_at_alert_rates.csv")
ppv_df.to_csv(ppv_path, index=False)
print("\nPPV/NPV at alert rates (first 12 rows):")
print(ppv_df.head(12).to_string(index=False))
print("\nSaved:", ppv_path)

In [ ]:
# =========================
# PEARSON / SPEARMAN OOF CORRELATIONS WITH CONTINUOUS DSI
# =========================
from scipy.stats import pearsonr, spearmanr

corr_rows = []

if "dsi_PM_total" in pm_day.columns:
    dsi = pm_day["dsi_PM_total"].values

    for f in oof_files:
        label = os.path.basename(f).replace("oof_", "").replace(".csv", "")
        df = pd.read_csv(f)

        # OOF file must have same row count as pm_day
        if len(df) != len(pm_day):
            continue

        for model_col in ["p_base", "p_full"]:
            if model_col not in df.columns:
                continue
            p = df[model_col].values
            mask = np.isfinite(dsi) & np.isfinite(p)
            if mask.sum() < 50:
                continue
            r, r_p = pearsonr(dsi[mask], p[mask])
            rho, rho_p = spearmanr(dsi[mask], p[mask])
            corr_rows.append({
                "label": label, "model": model_col,
                "pearson_r": r, "pearson_p": r_p,
                "spearman_rho": rho, "spearman_p": rho_p,
                "n": int(mask.sum()),
            })

    if corr_rows:
        corr_df = pd.DataFrame(corr_rows)
        corr_path = os.path.join(OUT_DIR, "oof_dsi_correlations.csv")
        corr_df.to_csv(corr_path, index=False)
        print("\nOOF-DSI correlations:")
        print(corr_df.to_string(index=False))
        print("\nSaved:", corr_path)
else:
    print("dsi_PM_total not found in features data")

In [ ]:
# =========================
# CONFOUND / SIGN-FLIP CHECKS
# =========================
print("=== Confound Checks ===")

checks = []

# 1) Is instability confounded with word count?
if "instability_cosdist" in pm_day.columns and "word_count" in pm_day.columns:
    for col in ["instability_cosdist", "instability_cosdist_within", "instability_cosdist_between"]:
        if col not in pm_day.columns:
            continue
        mask = pm_day["word_count"].notna() & pm_day[col].notna()
        if mask.sum() < 50:
            continue
        r = np.corrcoef(pm_day.loc[mask, "word_count"], pm_day.loc[mask, col])[0, 1]
        print(f"  corr(word_count, {col}) = {r:.4f}")
        checks.append({"check": f"corr_wc_{col}", "value": r})

# 2) Within-person PC directions: are they consistent?
for pc in ["PC1_within", "PC2_within", "PC3_within"]:
    if pc in pm_day.columns and CRISIS_COL in pm_day.columns:
        mask = pm_day[pc].notna() & pm_day[CRISIS_COL].notna()
        if mask.sum() < 50:
            continue
        r = np.corrcoef(pm_day.loc[mask, pc], pm_day.loc[mask, CRISIS_COL])[0, 1]
        print(f"  corr({pc}, {CRISIS_COL}) = {r:.4f}")
        checks.append({"check": f"corr_{pc}_crisis", "value": r})

# 3) Engagement and crisis
for col in ["log1p_wc", "wc_le1"]:
    if col in pm_day.columns and CRISIS_COL in pm_day.columns:
        mask = pm_day[col].notna() & pm_day[CRISIS_COL].notna()
        if mask.sum() < 50:
            continue
        r = np.corrcoef(pm_day.loc[mask, col], pm_day.loc[mask, CRISIS_COL])[0, 1]
        print(f"  corr({col}, {CRISIS_COL}) = {r:.4f}")
        checks.append({"check": f"corr_{col}_crisis", "value": r})

if checks:
    checks_df = pd.DataFrame(checks)
    checks_path = os.path.join(OUT_DIR, "confound_checks.csv")
    checks_df.to_csv(checks_path, index=False)
    print("\nSaved:", checks_path)